# D3RLPY Discrete Algorithms Integration Test

This notebook validates the full pipeline for the master thesis comparing:
- **Algorithms**: BC, CQL, Decision Transformer, TACR
- **Datasets**: CartPole, Atari Pong, Sepsis
- **Evaluation**: FQE for offline, rollouts for online

Running on GPU cluster to ensure infrastructure is ready for 36 experiments (4 algorithms × 3 datasets × 3 seeds).

## 1. Test Imports

In [ ]:
# Core algorithm imports
from d3rlpy.algos import DiscreteBC, DiscreteCQL, DiscreteDecisionTransformer, DiscreteTACR
from d3rlpy.ope import DiscreteFQE
from d3rlpy.datasets import get_cartpole, get_minari
import gymnasium as gym
import torch

print("✓ All imports successful")

## 2. GPU Check

In [ ]:
# Verify GPU availability
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU device: {torch.cuda.get_device_name(0)}")
    print(f"Device count: {torch.cuda.device_count()}")
else:
    print("WARNING: No GPU detected. Training will be slow.")

## 3. Dataset Loading

Test loading all three datasets for the thesis.

In [ ]:
# CartPole dataset
print("Loading CartPole dataset...")
cartpole_dataset, cartpole_env = get_cartpole()
print(f"  Episodes: {len(cartpole_dataset.episodes)}")
print(f"  Total steps: {sum(len(ep) for ep in cartpole_dataset.episodes)}")
print(f"  Action space: {cartpole_dataset.episodes[0].action_space}")
print("✓ CartPole loaded")

In [ ]:
# Atari Pong dataset via Minari
print("\nLoading Atari Pong dataset from Minari...")
try:
    # Common Minari dataset IDs for Atari Pong:
    # 'minari/pong-medium-v0', 'minari/pong-expert-v0', etc.
    pong_dataset, pong_env = get_minari('minari/pong-medium-v0')
    print(f"  Episodes: {len(pong_dataset.episodes)}")
    print(f"  Total steps: {sum(len(ep) for ep in pong_dataset.episodes)}")
    print("✓ Pong loaded")
except Exception as e:
    print(f"  ⚠ Pong not available yet: {e}")
    print("  Note: May need to download Minari dataset first")
    pong_dataset = None
    pong_env = None

In [ ]:
# Sepsis dataset placeholder
print("\nSepsis dataset (MIMIC-IV):")
print("  TODO: Implement datasets/sepsis_loader.py")
print("  This will load MIMIC-IV data in d3rlpy format")
sepsis_dataset = None

## 4. Algorithm Training Tests

Quick validation (2 steps each) that all algorithms can train on CartPole.

In [ ]:
# Test DiscreteBC
print("Testing DiscreteBC...")
bc = DiscreteBC()
bc.fit(cartpole_dataset, n_steps=2)
print("✓ DiscreteBC training works")

In [ ]:
# Test DiscreteCQL
print("\nTesting DiscreteCQL...")
cql = DiscreteCQL()
cql.fit(cartpole_dataset, n_steps=2)
print("✓ DiscreteCQL training works")

In [ ]:
# Test DiscreteDecisionTransformer
print("\nTesting DiscreteDecisionTransformer...")
dt = DiscreteDecisionTransformer(
    context_size=20,
    max_timestep=1000
)
dt.fit(cartpole_dataset, n_steps=2)
print("✓ DiscreteDecisionTransformer training works")

In [ ]:
# Test DiscreteTACR
print("\nTesting DiscreteTACR...")
tacr = DiscreteTACR(
    context_size=20,
    max_timestep=1000
)
tacr.fit(cartpole_dataset, n_steps=2)
print("✓ DiscreteTACR training works")

## 5. FQE Evaluation Test

Validate that Fitted Q Evaluation works (needed for offline Sepsis evaluation).

In [ ]:
# Test DiscreteFQE
print("Testing DiscreteFQE...")
# Use the trained BC policy for evaluation
fqe = DiscreteFQE(algo=bc)
fqe.fit(cartpole_dataset, n_steps=2)
print("✓ DiscreteFQE training works")

# Estimate policy value
estimated_value = fqe.predict_value(cartpole_dataset.episodes[0].observations[:10])
print(f"Sample Q-values shape: {estimated_value.shape}")

## 6. Environment Rollout Test

Validate online evaluation capability (needed for CartPole and Pong benchmarks).

In [ ]:
# Test environment rollout with trained BC policy
print("Testing environment rollout...")
env = gym.make('CartPole-v1')
observation, info = env.reset()

total_reward = 0
for step in range(10):
    action = bc.predict([observation])[0]
    observation, reward, terminated, truncated, info = env.step(action)
    total_reward += reward
    if terminated or truncated:
        break

env.close()
print(f"✓ Rollout completed: {step+1} steps, total reward = {total_reward}")

## Summary

If all cells above ran successfully, the infrastructure is ready for:
1. ✅ Training all 4 discrete algorithms (BC, CQL, DT, TACR)
2. ✅ Loading CartPole dataset
3. ⚠️  Loading Pong dataset (may need Minari setup)
4. 📝 Loading Sepsis dataset (requires sepsis_loader.py implementation)
5. ✅ FQE offline evaluation
6. ✅ Environment rollouts for online evaluation

**Next steps:**
- Implement `datasets/sepsis_loader.py` for MIMIC-IV data
- Create unified `training/train.py` script
- Set up SLURM templates for batch submission
- Launch 36 experiments (4 algorithms × 3 datasets × 3 seeds)